# Masking Study Comparing 1D and 2D Fits

The objective of this study is to reconcile one-dimensional and two-dimensional fits of the relative intensity skymaps for the IceTop eleven-year analysis. 

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from glob import glob

import numpy as np
import healpy as hp
import matplotlib.pyplot as plt

from Fit2d import multipoleFit2d, returnRI, multipoleFit1D

## Load Skymaps

In [4]:
# Load in maps

relint_files = {}
relerr_files = {}
root = '/data/ana/CosmicRay/Anisotropy/IceTop/ITpass2/output/sidereal_unblinded'

tiers = [i for i in range(1, 5)]

for tier in tiers:
    files = glob(f'{root}/tier{tier}/reconstruction/relintensityiter/combined*.fits.gz')
    relint_files[tier] = sorted(files)[-1]

    files = glob(f'{root}/tier{tier}/reconstruction/significanceiter/significance*.fits.gz')
    relerr_files[tier] = sorted(files)[-1]

print(f'Root directory: {root}')
print('Relative intensity files:')
for tier, map_file in relint_files.items():
    print(f'  Tier {tier} : {{root}}{map_file.replace(root,"")}')

print('Relative error files:')
for tier, map_file in relerr_files.items():
    print(f'  Tier {tier} : {{root}}{map_file.replace(root,"")}')

Root directory: /data/ana/CosmicRay/Anisotropy/IceTop/ITpass2/output/sidereal_unblinded
Relative intensity files:
  Tier 1 : {root}/tier1/reconstruction/relintensityiter/combined_t1_iteration04.fits.gz
  Tier 2 : {root}/tier2/reconstruction/relintensityiter/combined_t2_iteration20.fits.gz
  Tier 3 : {root}/tier3/reconstruction/relintensityiter/combined_t3_iteration20.fits.gz
  Tier 4 : {root}/tier4/reconstruction/relintensityiter/combined_t4_iteration20.fits.gz
Relative error files:
  Tier 1 : {root}/tier1/reconstruction/significanceiter/significance_t1_iteration04.fits.gz
  Tier 2 : {root}/tier2/reconstruction/significanceiter/significance_t2_iteration20.fits.gz
  Tier 3 : {root}/tier3/reconstruction/significanceiter/significance_t3_iteration20.fits.gz
  Tier 4 : {root}/tier4/reconstruction/significanceiter/significance_t4_iteration20.fits.gz


In [5]:
# Calculate 2D fit with standard masking
def get_2d_fit_vals(relint_files, relerr_files, decmin, decmax):

    tiers = sorted(relint_files.keys())
    amp, phase, amp_err, phase_err = np.zeros((4, len(tiers)))

    for i, tier in enumerate(tiers):
    
        data, bg, relint = hp.read_map(relint_files[tier], range(3))
        relerr,_,_ = hp.read_map(relerr_files[tier], range(3))
        
        popt, perr, chi2, ndof, pvalue = multipoleFit2d(relint, relerr, 3, decmin=decmin, decmax=decmax)
    
        a = np.reshape(popt, (-1,2))
        amp[i], phase[i] = a[0]
    
        # Amplitude/phase correction for negative signs
        if amp[i] < 0:
            amp[i] *= -1
            phase[i] += np.pi
        if phase[i] < 0:
            phase[i] += 2*np.pi
        phase[i] %= 2*np.pi
        
        e = np.reshape(perr, (-1,2))
        amp_err[i], phase_err[i] = e[0]

    return amp, phase, amp_err, phase_err

In [7]:
# Print values for the paper
decmin = -90.
decmax = -35.

amp_2d, phase_2d, amp_err_2d, phase_err_2d = get_2d_fit_vals(relint_files, relerr_files, decmin, decmax)

In [8]:
for i, tier in enumerate(tiers):
    print(f'Tier {tier}:')
    print(f'  Amplitude (2D): {amp_2d[i]:.5f} ± {amp_err_2d[i]:.5f}')
    print(f'  Phase (2D):     {np.degrees(phase_2d[i]):.3f} ± {np.degrees(phase_err_2d[i]):.3f}')

Tier 1:
  Amplitude (2D): 0.00059 ± 0.00013
  Phase (2D):     237.759 ± 12.852
Tier 2:
  Amplitude (2D): 0.00232 ± 0.00015
  Phase (2D):     268.493 ± 3.756
Tier 3:
  Amplitude (2D): 0.00261 ± 0.00019
  Phase (2D):     278.920 ± 4.116
Tier 4:
  Amplitude (2D): 0.00178 ± 0.00054
  Phase (2D):     303.336 ± 17.523
